# CORD Receipt → JSON


In [ ]:
# install the requirements
!pip install -r ../requirements.txt
!pip install -U ipywidgets==7.7.1

# Dataset Prep

In [ ]:
from datasets import load_dataset

# Load CORD-v2 receipt dataset (all 3 splits)
dataset = load_dataset("naver-clova-ix/cord-v2")

train_dataset = dataset["train"]
validation_dataset = dataset["validation"]
test_dataset = dataset["test"]

print(f"Train samples:      {len(train_dataset)}")
print(f"Validation samples: {len(validation_dataset)}")
print(f"Test samples:       {len(test_dataset)}")
print(f"\nFeatures: {train_dataset.features}")


In [ ]:
import random
import json
import base64
from io import BytesIO
from IPython.display import display, HTML

def show_random_samples(dataset, n=4):
    indices = random.sample(range(len(dataset)), n)

    html_parts = ['<div style="display:flex; flex-wrap:wrap; gap:16px;">']

    for idx in indices:
        sample = dataset[idx]
        gt = json.loads(sample["ground_truth"])

        # Keep only gt_parse
        gt_parse = gt.get("gt_parse", {})

        # Convert PIL image to base64 PNG
        buf = BytesIO()
        sample["image"].save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()

        json_str = json.dumps(gt_parse, indent=2)

        html_parts.append(f"""
        <div style="border:1px solid #ccc; padding:10px; border-radius:8px; max-width:340px;">
            <p style="font-weight:bold; margin:0 0 6px 0;">Sample #{idx}</p>
            <img src="data:image/png;base64,{b64}"
                 style="max-width:300px; width:100%; display:block; border-radius:4px;"/>
            <pre style="font-size:9px; background:#f5f5f5; padding:8px; border-radius:4px;
                        max-height:220px; overflow-y:auto; white-space:pre-wrap;
                        margin-top:8px; font-family:monospace;">{json_str}</pre>
        </div>
        """)

    html_parts.append("</div>")
    display(HTML("".join(html_parts)))


In [ ]:
show_random_samples(train_dataset, n=3)

From this datasets we do not need most of the label, except json in the gt_parse section

In [61]:
from _cord_preprocess import preprocess_cord
train_dataset, validation_dataset, test_dataset = preprocess_cord(dataset)

mixed fields (-> list-of-str), frozen to mixed_fields.json:
   menu.cnt
   menu.discountprice
   menu.nm
   menu.num
   sub_total.discount_price
   sub_total.etc
   sub_total.subtotal_price
   sub_total.tax_price
   total.cashprice
   total.changeprice
   total.creditcardprice
   total.total_price
sizes: train=800 val=100 test=100


In [ ]:
from _cord_preprocess import test
test(train_dataset[1])

ImportError: cannot import name 'test' from '_cord_preprocess' (/workspace/fine-tune-receipe-pdf/notebooks/_cord_preprocess.py)

In [62]:
from _receipt_schema import assert_schema_valid
assert_schema_valid(train_dataset, "train")
assert_schema_valid(validation_dataset, "validation")
assert_schema_valid(test_dataset, "test")

[train] reject: 2 validation errors for Receipt
menu.1.sub
  Input should be a valid list [type=list_type, input_value={'nm': 'WELL DONE'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type
menu.2.sub
  Input should be a valid list [type=list_type, input_value={'nm': 'MEDIUM WELL'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type
  label={"menu": [{"nm": ["SPGTHY BOLOGNASE"], "cnt": ["1"], "price": "58,000"}, {"nm": ["PEPPER AUS"], "cnt": ["1"], "price": "165,000", "sub": {"nm": "WELL DONE"}}, {"nm": ["WAGYU RIBEYE"], "cnt": ["1"], "p
[train] reject: 5 validation errors for Receipt
menu.0.sub
  Input should be a valid list [type=list_type, input_value={'nm': '=*LARGE*==', 'cnt': '2'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type
menu.1.sub
  Input should be a valid list [type=list_type, input_value={'nm': '=*MEDIUM*='}, input_type=dict]
    Fo

9

# Model Prep

Download the model

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = snapshot_download(
    repo_id="Qwen/Qwen3-VL-2B-Instruct",
    local_dir="../models/qwen3vl-2b",
)
print("downloaded to", MODEL_ID)

Import mode to code

In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2",  # if flash-attn is installed
)
processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=1200*28*28)

# REQUIRED for the prefix-based prompt masking below to land on the right tokens
processor.tokenizer.padding_side = "right"

Inspect the model architecture

In [ ]:
import re
seen = set()
for name, mod in model.named_modules():
    if mod.__class__.__name__ == "Linear" and (".visual" in name or "vision" in name):
        key = re.sub(r"\.\d+\.", ".N.", name)
        if key not in seen:
            seen.add(key)
            print(name)
# Expect attention like ...attn.qkv / ...attn.proj and MLP gate/up/down_proj.

In [ ]:
for name, mod in model.named_modules():
    if mod.__class__.__name__ == "Linear":
        tower = "VISION" if (".visual" in name or "vision" in name) else "LANG  "
        print(f"[{tower}] {name}")

Set up LoRA config

In [ ]:
from peft import LoraConfig

LM_TARGETS     = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]
VISION_TARGETS = ["qkv", "proj", "linear_fc1", "linear_fc2"]

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules= LM_TARGETS,
)

In [ ]:
FIXED_PROMPT = """Extract the receipt from the image into JSON. Your ouput should contain ONLY correct JSON
"""

# image placeholder id; verify the attribute name on your build:
IMAGE_TOKEN_ID = getattr(model.config, "image_token_id", None)

def collate_fn(examples):
    # each example: {"image": <PIL.Image>, "label": "<output text>"}
    full_texts, prompt_texts, images = [], [], []
    for ex in examples:
        user_turn = [{"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": FIXED_PROMPT},
        ]}]
        full_msg = user_turn + [{"role": "assistant", "content": [
            {"type": "text", "text": ex["label"]},
        ]}]
        full_texts.append(
            processor.apply_chat_template(full_msg, tokenize=False, add_generation_prompt=False))
        prompt_texts.append(
            processor.apply_chat_template(user_turn, tokenize=False, add_generation_prompt=True))
        images.append(ex["image"])

    batch = processor(text=full_texts, images=images, padding=True, return_tensors="pt")
    labels = batch["input_ids"].clone()

    # 1) mask the fixed prompt + image tokens per sample (right padding => prompt at the front).
    #    Re-run processor on prompt-only WITH the image so image-token expansion is counted.
    for i, (p_text, img) in enumerate(zip(prompt_texts, images)):
        plen = processor(text=[p_text], images=[img], return_tensors="pt")["input_ids"].shape[1]
        labels[i, :plen] = -100

    # 2) mask padding
    labels[labels == processor.tokenizer.pad_token_id] = -100
    # 3) defensively mask any image placeholder tokens that survive in the completion region
    if IMAGE_TOKEN_ID is not None:
        labels[labels == IMAGE_TOKEN_ID] = -100

    batch["labels"] = labels
    return batch

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="../outputs/qwen3vl-2B-lora-vision-frozen-lsg-out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=5,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_steps=50,
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    eval_strategy="steps",
    eval_steps=50,
    per_device_eval_batch_size=2,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,       # TRL applies LoRA + wires grads for you
    processing_class=processor,    # full processor, not processor.tokenizer
)

In [ ]:
trainer.train()
trainer.save_model("../outputs/qwen3vl-2B-lora-vision-frozen-lsg-adapter") 